In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Simplified Conditional Autoregressive Generation

This notebook follows the same generation process used in `example_conditional_autoregressive_generator_molecule_generation.ipynb`, adapted to the new simplified implementation.

In [ ]:
%config InlineBackend.figure_format = 'retina'

import random
import time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from abstractgraph.display import display_graphs

from abstractgraph.operators import (
    add,
    compose,
    connected_component,
    edge_complement,
    intersection_edges,
    low_cut_partition,
    merge,
    name,
    remove_redundant_mapped_subgraphs,
)
from abstractgraph.hashing import hash_graph
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph_generative.conditional import (
    ConditionalAutoregressiveGenerator,
)
from abstractgraph_generative.conditional_batch import (
    ConditionalAutoregressiveGraphsGenerator,
)


In [ ]:
from abstractgraph.graphs import AbstractGraph, graph_to_abstract_graph
from abstractgraph.display import display, display_graph, display_mappings, display_decomposition_graph, decomposition_to_graph
from abstractgraph.labels import graph_hash_label_function_factory
from abstractgraph.operators import *
from abstractgraph import ArtificialGraphDatasetConstructor

def draw(graph, df, nbits=10, label_mode="operator"):
    display_decomposition_graph(df)
    ag = graph_to_abstract_graph(graph, decomposition_function=df, nbits=nbits, label_mode=label_mode)
    display(ag, size=(12,6))
    display_mappings(ag, n_elements_per_row=10)

## 1) Build a synthetic pos/neg graph dataset

In [ ]:
def offset_neg_graphs(graphs, targets, offset=10):
    out_graphs = []
    for graph, target in zip(graphs, targets):
        graph = graph.copy()
        if target == 0:
            for u in graph.nodes():
                graph.nodes[u]["label"] += offset
        out_graphs.append(graph)
    return out_graphs, targets


def select_pos_neg(sampled_graphs, sampled_targets, n_lines=3, n_graphs_per_line=12):
    import random

    k = n_graphs_per_line * n_lines
    pos_candidates = [
        sampled_graph
        for sampled_graph, sampled_target in zip(sampled_graphs, sampled_targets)
        if sampled_target == 1
    ]
    neg_candidates = [
        sampled_graph
        for sampled_graph, sampled_target in zip(sampled_graphs, sampled_targets)
        if sampled_target != 1
    ]
    sampled_pos_graphs = random.sample(pos_candidates, k=min(k, len(pos_candidates)))
    sampled_neg_graphs = random.sample(neg_candidates, k=min(k, len(neg_candidates)))
    return sampled_pos_graphs, sampled_neg_graphs


dataset_size = 100
alphabet_size = 4
size = 10
graph_types = ["path", "tree", "cycle", "degree", "regular", "dense"]

graphs, targets = ArtificialGraphDatasetConstructor(
    graph_generator_target_type_pos="cycle",
    graph_generator_context_type_pos="cycle",
    graph_generator_target_type_neg="path",
    graph_generator_context_type_neg="path",
    target_size_pos=size,
    context_size_pos=size,
    n_link_edges_pos=1,
    alphabet_size_pos=alphabet_size,
    target_size_neg=size,
    context_size_neg=size,
    n_link_edges_neg=1,
    alphabet_size_neg=alphabet_size,
).sample(dataset_size // 2)

graphs, targets = offset_neg_graphs(graphs, targets, offset=alphabet_size + 1)
targets = np.array(targets)

print("#graphs:%d" % (len(graphs)))

n_graphs_per_line = 10
n_lines = 2
num_graphs = n_lines * n_graphs_per_line
display_pos_graphs, display_neg_graphs = select_pos_neg(
    graphs,
    targets,
    n_lines=n_lines,
    n_graphs_per_line=n_graphs_per_line,
)

display_graph_size = 2
_ = display_graphs(
    display_neg_graphs,
    n_graphs_per_line=n_graphs_per_line,
    size=(display_graph_size, display_graph_size),
)
_ = display_graphs(
    display_pos_graphs,
    n_graphs_per_line=n_graphs_per_line,
    size=(display_graph_size, display_graph_size),
)

# Keep explicit dataset-level pools separate from display samples.
all_graphs = graphs
all_targets = targets
full_pos_graphs = [graph for graph, target in zip(all_graphs, all_targets) if target == 1]
full_neg_graphs = [graph for graph, target in zip(all_graphs, all_targets) if target != 1]
print("dataset size:", len(all_graphs))
print("positive pool:", len(full_pos_graphs), "negative pool:", len(full_neg_graphs))


In [ ]:
def _fallback_draw_graphs(graphs, cols=4, title='graphs'):
    rows = (len(graphs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.2 * rows))
    axes = axes.ravel() if hasattr(axes, 'ravel') else [axes]
    for i in range(rows * cols):
        ax = axes[i]
        if i < len(graphs):
            g = graphs[i]
            pos = nx.spring_layout(g, seed=1)
            node_labels = nx.get_node_attributes(g, 'label')
            nx.draw(g, pos=pos, ax=ax, with_labels=True, labels=node_labels, node_size=300, font_size=8)
            ax.set_title(f'n={g.number_of_nodes()} e={g.number_of_edges()}', fontsize=9)
        else:
            ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_graphs(graphs, cols=4, title='graphs'):
    if not graphs:
        print('No graphs to display')
        return
    print(title)
    try:
        return display_graphs(graphs, n_graphs_per_line=cols, size=(2, 2))
    except Exception:
        return _fallback_draw_graphs(graphs, cols=cols, title=title)

## 2) Same generation process helpers (`same_image` / `nn`)

In [ ]:
def graph_signature_vector(g: nx.Graph) -> np.ndarray:
    labels = nx.get_node_attributes(g, 'label')
    counts = {k: 0 for k in ['C', 'N', 'O', 'S']}
    for v in labels.values():
        counts[v] = counts.get(v, 0) + 1
    return np.array([
        g.number_of_nodes(),
        g.number_of_edges(),
        counts.get('C', 0),
        counts.get('N', 0),
        counts.get('O', 0),
        counts.get('S', 0),
    ], dtype=float)


def group_graphs_by_interpretation_hash(graphs, decomposition_function, nbits):
    buckets = {}
    for g in graphs:
        ag = graph_to_abstract_graph(
            g,
            decomposition_function=decomposition_function,
            nbits=nbits,
            label_mode='operator',
        )
        interpretation_hash = hash_graph(ag.interpretation_graph)
        buckets.setdefault(interpretation_hash, []).append(g)
    return buckets


def select_interpretation_group(buckets, min_group_size, rng):
    if not buckets:
        return [], True
    eligible = [grp for grp in buckets.values() if len(grp) >= min_group_size]
    if eligible:
        return rng.choice(eligible), False
    return max(buckets.values(), key=len), True


def select_nn_group(graphs, seed_graph, group_size):
    seed_vec = graph_signature_vector(seed_graph)
    scored = []
    for g in graphs:
        v = graph_signature_vector(g)
        d = float(np.linalg.norm(v - seed_vec))
        scored.append((d, g))
    scored.sort(key=lambda x: x[0])
    return [g for _, g in scored[:group_size]]


def _graph_size_summary(graphs):
    if not graphs:
        return 'empty'
    nodes = [g.number_of_nodes() for g in graphs]
    edges = [g.number_of_edges() for g in graphs]
    return (
        f"nodes(min/mean/max)=({min(nodes)}/{np.mean(nodes):.1f}/{max(nodes)}) "
        f"edges(min/mean/max)=({min(edges)}/{np.mean(edges):.1f}/{max(edges)})"
    )


def generate_graphs(
    graphs,
    *,
    decomposition_function,
    nbits,
    mode='same_image',
    generator_size=7,
    n_samples=7,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    random_seed=0,
    feasibility_estimator=None,
    max_backtracks=12000,
    max_attempts_per_sample=48,
    max_total_attempts=None,
    debug=True,
    return_stats=False,
):
    rng = random.Random(random_seed)
    if len(graphs) < generator_size:
        raise ValueError('graphs size must be >= generator_size')

    if debug:
        print('' + '=' * 100)
        print(f'[DEBUG] mode={mode} nbits={nbits} generator_size={generator_size} n_samples={n_samples}')
        print(f'[DEBUG] radii base={base_cut_radius} interpretation={interpretation_cut_radius}')
        print(f'[DEBUG] search max_backtracks={max_backtracks} max_attempts_per_sample={max_attempts_per_sample}')

    if mode == 'same_image':
        buckets = group_graphs_by_interpretation_hash(graphs, decomposition_function, nbits)
        if debug:
            bucket_sizes = sorted((len(v) for v in buckets.values()), reverse=True)
            print(f'[DEBUG] image-hash buckets={len(bucket_sizes)} top10={bucket_sizes[:10]}')
        generator_graphs, used_fallback = select_interpretation_group(buckets, generator_size, rng)
        if used_fallback:
            print('[DEBUG] same_image fallback: largest image-hash bucket used')
        if len(generator_graphs) > generator_size:
            rng.shuffle(generator_graphs)
            generator_graphs = generator_graphs[:generator_size]
    elif mode == 'nn':
        seed_graph = rng.choice(graphs)
        if debug:
            print(f'[DEBUG] nn seed graph: nodes={seed_graph.number_of_nodes()} edges={seed_graph.number_of_edges()}')
        generator_graphs = select_nn_group(graphs, seed_graph, generator_size)
    else:
        raise ValueError("mode must be one of {'same_image', 'nn'}")

    if debug:
        print(f'[DEBUG] generator set size={len(generator_graphs)}')
        print(f'[DEBUG] generator set stats: {_graph_size_summary(generator_graphs)}')

    generator = ConditionalAutoregressiveGenerator(
        decomposition_function=decomposition_function,
        nbits=nbits,
        feasibility_estimator=feasibility_estimator,
        base_cut_radius=base_cut_radius,
        interpretation_cut_radius=interpretation_cut_radius,
        n_jobs=-1,
        debug=debug,
    )

    t0 = time.time()
    generator.fit(generator_graphs)
    fit_s = time.time() - t0

    t1 = time.time()
    strict_samples = generator.generate(
        n_samples=n_samples,
        random_state=random_seed,
        max_backtracks=max_backtracks,
        max_attempts_per_sample=max_attempts_per_sample,
        max_total_attempts=max_total_attempts,
    )
    gen_s = time.time() - t1

    samples = strict_samples
    strict_count = len(strict_samples)

    print(f'mode={mode} generator_set={len(generator_graphs)} fit={fit_s:.2f}s generate={gen_s:.2f}s out={strict_count}')

    if debug:
        print(f'[DEBUG] strict output stats: {_graph_size_summary(strict_samples)}')

    if return_stats:
        stats = {
            'strict_generated': strict_count,
            'final_generated': len(samples),
            'relaxed_used': False,
            'generator_set_size': len(generator_graphs),
            'fit_seconds': fit_s,
            'generate_seconds': gen_s,
            'base_cut_radius': base_cut_radius,
            'interpretation_cut_radius': interpretation_cut_radius,
            'mode': mode,
        }
        return samples, generator_graphs, stats

    return samples, generator_graphs


---

In [ ]:
from abstractgraph.operators import *


core_df = compose(name('cor'),betweenness_centrality_hop_split(n_hops=2))
decomposition_function = compose(intersection_edges(),remove_redundant_mapped_subgraphs(), core_df) 

df = add(compose(name('cyc'), cycle()), compose(name('tree'), tree()))
decomposition_function = compose(intersection_edges(), df)

nbits = 14

from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator, FeasibilityEstimatorNumberOfNodesInRange
from abstractgraph.operators import *
min_size = min(len(graph) for graph in graphs)
max_size = max(len(graph) for graph in graphs)
fe0 = FeasibilityEstimatorNumberOfNodesInRange(min_size=min_size, max_size=max_size)
df = compose(neighborhood(radius=2), unlabel())
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19, parallel=True, backend="dill")
df = neighborhood(radius=1)
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19, parallel=True, backend="dill")
df = cycle()
fe3 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19, parallel=True, backend="dill")
feasibility_estimators = [fe0, fe2]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

#feasibility_estimator= None

draw(display_pos_graphs[1], decomposition_function, nbits)

In [ ]:
%%time
nn_samples, nn_generator_set = generate_graphs(
    full_pos_graphs,
    decomposition_function=decomposition_function,
    nbits=nbits,
    mode='nn',
    generator_size=7*3,
    n_samples=7*10,
    base_cut_radius=0,
    interpretation_cut_radius=0,
    random_seed=None,
    feasibility_estimator=feasibility_estimator,
    max_backtracks=12000,
    max_attempts_per_sample=48,
    debug=True,
)

from abstractgraph.hashing import hash_graph, GraphHashDeduper
_ = show_graphs(nn_generator_set, cols=7, title=f'Generator set (nn) {len(nn_generator_set)} samples')
novel_samples = GraphHashDeduper().fit(nn_generator_set).filter(nn_samples)
_ = show_graphs(novel_samples, cols=7, title=f'Novel {len(novel_samples)} samples')
unique_samples = GraphHashDeduper().fit_filter(nn_samples)
_ = show_graphs(unique_samples, cols=7, title=f'Unique {len(unique_samples)} samples')
_ = show_graphs(nn_samples, cols=7, title=f'Generated (nn) {len(nn_samples)} samples')